In [1]:
# =============================================================================
# Insider Threat Behavioral Intelligence System
# Notebook : 04_risk_scoring.ipynb
# =============================================================================

# Module 04: Composite Employee Risk Scoring Engine

This notebook computes unified 0-100 employee risk scores by combining multi-model consensus predictions with weighted anomaly signals and authoritative risk level categorizations:
- **CRITICAL**: Risk Score 75.0 – 100.0 (Score >= 75.0)
- **HIGH**: Risk Score 50.0 – 74.9 (Score >= 50.0 and < 75.0)
- **MEDIUM**: Risk Score 25.0 – 49.9 (Score >= 25.0 and < 50.0)
- **LOW**: Risk Score 0.0 – 24.9 (Score < 25.0)

### Pipeline Overview:
ML Anomaly Detection (7 Models) $\rightarrow$ Multi-Model Consensus $\rightarrow$ Composite Risk Scoring $\rightarrow$ CERT Layer 2 Population Verification

In [2]:
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [3]:
PROJECT_ROOT = Path("..").resolve()
CONSENSUS_FILE = PROJECT_ROOT / "reports" / "consensus_predictions.csv"
EXPORT_DIR = PROJECT_ROOT / "datasets" / "exports"
PLOT_DIR = PROJECT_ROOT / "plots"

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

if CONSENSUS_FILE.exists():
    df = pd.read_csv(CONSENSUS_FILE)
else:
    print("Creating fallback consensus dataset for risk scoring.")
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({'user': [f'USR{i:04d}' for i in range(n)]})
    cols = ['pred_isolation_forest', 'pred_one_class_svm', 'pred_lof', 'pred_elliptic_envelope', 'pred_pca', 'pred_dbscan', 'pred_kmeans']
    for c in cols:
        df[c] = np.random.choice([1, -1], size=n, p=[0.94, 0.06])
    df['consensus_votes'] = (df[cols] == -1).sum(axis=1)

print(f"Loaded consensus data for {len(df)} employees.")
df.head()

## 1. Weighted Multi-Model Risk Score Computation & Authoritative Thresholds

In [4]:
# Define 7-Model Weighting Scheme
model_weights = {
    "pred_isolation_forest": 0.20,
    "pred_one_class_svm": 0.15,
    "pred_lof": 0.15,
    "pred_elliptic_envelope": 0.10,
    "pred_pca": 0.10,
    "pred_dbscan": 0.15,
    "pred_kmeans": 0.15
}

# Calculate raw weighted anomaly vote score
df['raw_weighted_score'] = 0.0
for col, weight in model_weights.items():
    if col in df.columns:
        # Convert prediction (-1 anomaly, 1 normal) to binary (1 anomaly, 0 normal)
        anom_binary = np.where(df[col] == -1, 1.0, 0.0)
        df['raw_weighted_score'] += anom_binary * weight

# Scale to 0 - 100 Range with Min-Max Scaling
min_s = df['raw_weighted_score'].min()
max_s = df['raw_weighted_score'].max()

if max_s > min_s:
    df['risk_score'] = ((df['raw_weighted_score'] - min_s) / (max_s - min_s) * 100).round(2)
else:
    df['risk_score'] = (df['raw_weighted_score'] * 100).round(2)

# Assign Authoritative Application Risk Boundaries:
# LOW: 0.0 - 24.9 | MEDIUM: 25.0 - 49.9 | HIGH: 50.0 - 74.9 | CRITICAL: 75.0 - 100.0
def assign_risk_level(score):
    if score >= 75.0:
        return 'CRITICAL'
    elif score >= 50.0:
        return 'HIGH'
    elif score >= 25.0:
        return 'MEDIUM'
    else:
        return 'LOW'

df['risk_level'] = df['risk_score'].apply(assign_risk_level)

risk_distribution = df['risk_level'].value_counts()
print("Risk Level Breakdown (Authoritative Boundaries):")
print(risk_distribution)

## 2. Top At-Risk Employees Analysis

In [5]:
top_at_risk = df.sort_values(by='risk_score', ascending=False).head(10)

plt.figure(figsize=(10, 5))
colors = ['#dc3545' if r == 'CRITICAL' else '#fd7e14' if r == 'HIGH' else '#ffc107' if r == 'MEDIUM' else '#28a745' for r in top_at_risk['risk_level']]
plt.barh(top_at_risk['user'][::-1], top_at_risk['risk_score'][::-1], color=colors[::-1])
plt.xlabel("Risk Score (0 - 100)", fontsize=12)
plt.ylabel("Employee ID", fontsize=12)
plt.title("Top 10 At-Risk Employees", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOT_DIR / "top_at_risk_employees.png", dpi=300)
plt.show()

top_at_risk[['user', 'risk_score', 'risk_level', 'consensus_votes']]

## 3. CERT Layer 2 Behavioral Verification & Export

**System Architecture Disambiguation:**
1. **ML Anomaly Detection**: Unsupervised multi-model outlier scoring.
2. **Multi-Model Consensus**: Ensemble agreement voting across the 7 models.
3. **Composite Risk Scoring**: Weighted score mapping to authoritative risk levels ($0.0-24.9$ LOW, $25.0-49.9$ MEDIUM, $50.0-74.9$ HIGH, $75.0-100.0$ CRITICAL).
4. **CERT Layer 2 Verification**: Independent population-level statistical baseline validation (P90/P95 criteria across 6 CERT behavioral dimensions: Temporal, Device/USB, Multi-PC, Web, Email, Overall Volume).

In [6]:
output_csv = EXPORT_DIR / "employee_final_risk_report.csv"
output_parquet = EXPORT_DIR / "employee_final_risk_report.parquet"

df.to_csv(output_csv, index=False)
df.to_parquet(output_parquet, index=False)

print(f"Final Risk Report CSV saved: {output_csv}")
print(f"Final Risk Report Parquet saved: {output_parquet}")